Establecer una conexión entre Python y la base de datos Sakila.

In [224]:
from sqlalchemy import create_engine
import pandas as pd

# Datos de conexión
user = "root"
password = "Ozz4tr1s#2026"
host = "localhost"
database = "sakila"

# Crear el engine
engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}/{database}")

# Probar la conexión leyendo una tabla
df_total = pd.read_sql("SELECT * FROM rental LIMIT 10;", engine)
df_total.shape


(10, 7)

In [225]:
df_total.head()

,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1,2005-05-24 22:53:30,367,130,2005-05-26 22:04:30,1,2006-02-15 21:30:53
1,2,2005-05-24 22:54:33,1525,459,2005-05-28 19:40:33,1,2006-02-15 21:30:53
2,3,2005-05-24 23:03:39,1711,408,2005-06-01 22:12:39,1,2006-02-15 21:30:53
3,4,2005-05-24 23:04:41,2452,333,2005-06-03 01:43:41,2,2006-02-15 21:30:53
4,5,2005-05-24 23:05:21,2079,222,2005-06-02 04:33:21,1,2006-02-15 21:30:53


Vamos a ver el rango total de fechas, pero primero quito la lomitacion de 10

In [226]:
df_total = pd.read_sql("SELECT * FROM rental;", engine)
df_total['rental_date'].min(), df_total['rental_date'].max()
df_total.shape



(16044, 7)

In [227]:
df_total['rental_date'].dt.to_period('M').unique() # esta formula nos dice los meses años que vienen en la base de datos


<PeriodArray>
['2005-05', '2005-06', '2005-07', '2005-08', '2006-02']
Length: 5, dtype: period[M]

In [228]:
df_total['rental_date'].dt.to_period('M').value_counts().sort_index()


rental_date
2005-05    1156
2005-06    2311
2005-07    6709
2005-08    5686
2006-02     182
Freq: M, Name: count, dtype: int64

Escriba una función de Python llamada rentals_month que recupere datos de alquiler para un mes y año determinados (pasados ​​como parámetros) de la base de datos Sakila como un DataFrame de Pandas. La función debe tomar tres parámetros: engine: un objeto que representa el motor de conexión a la base de datos que se utilizará para establecer una conexión con la base de datos Sakila. month: un entero que representa el mes para el que se recuperarán los datos de alquiler. year: un entero que representa el año para el que se recuperarán los datos de alquiler  
La función debe ejecutar una consulta SQL para recuperar los datos de alquiler del mes y año especificados de la tabla de alquileres en la base de datos Sakila, y devolverlos como un DataFrame de pandas.

In [229]:
def rentals_month(engine, month, year): # engie para el return month, year para hacer la query de SQL
    
    query = f"""
        SELECT *
        FROM rental
        WHERE MONTH(rental_date) = {month} # de 1 a 12
          AND YEAR(rental_date) = {year}; # con 4 cifras
    """
    
    return pd.read_sql(query, engine) #  engine conectada a Sakila con 
                                        # (function) def read_sql(sql: _SQLStatement,  con: _SQLConnection,-...)

In [230]:
df_rentals_5_2005 = rentals_month(engine,5,2005) 
print(df_rentals_5_2005.shape)
df_rentals_5_2005.head()

(1156, 7)


,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1,2005-05-24 22:53:30,367,130,2005-05-26 22:04:30,1,2006-02-15 21:30:53
1,2,2005-05-24 22:54:33,1525,459,2005-05-28 19:40:33,1,2006-02-15 21:30:53
2,3,2005-05-24 23:03:39,1711,408,2005-06-01 22:12:39,1,2006-02-15 21:30:53
3,4,2005-05-24 23:04:41,2452,333,2005-06-03 01:43:41,2,2006-02-15 21:30:53
4,5,2005-05-24 23:05:21,2079,222,2005-06-02 04:33:21,1,2006-02-15 21:30:53


In [231]:
df_rentals_2_2006 = rentals_month(engine,2,2006) 
print(df_rentals_2_2006.shape)
df_rentals_2_2006.head()

(182, 7)


,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,11496,2006-02-14 15:16:03,2047,155,None,1,2006-02-15 21:30:53
1,11541,2006-02-14 15:16:03,2026,335,None,1,2006-02-15 21:30:53
2,11563,2006-02-14 15:16:03,1545,83,None,1,2006-02-15 21:30:53
3,11577,2006-02-14 15:16:03,4106,219,None,2,2006-02-15 21:30:53
4,11593,2006-02-14 15:16:03,817,99,None,1,2006-02-15 21:30:53


Desarrolle una función de Python llamada Rental_count_month que tome el DataFrame proporcionado por Rentals_month como entrada junto con el mes y el año y devuelva un nuevo DataFrame que contenga la cantidad de alquileres realizados por cada customer_id durante el mes y año seleccionados.  

La función también debe incluir el mes y el año como parámetros y usarlos para nombrar la nueva columna según el mes y el año, por ejemplo, si el mes de entrada es 05 y el año es 2005, el nombre de la columna debe ser "alquileres_05_2005".  

Sugerencia: considere utilizar pandas groupby()


In [232]:
def rental_counth_month(df, month, year):

    nombre_columna_nueva = "alquileres"+"_"+ str(month).zfill(2) + "_" + str(year)
    nombre_columna_nueva
    serie_alquileres = df.groupby('customer_id').size()
    df_final = serie_alquileres.reset_index()
    df_final.columns = ["customer_id",nombre_columna_nueva ]
    df_final.head()

    return df_final

In [233]:
df_alquileres_5_2005 = rental_counth_month(df_rentals_5_2005,5,2005) 
df_alquileres_5_2005.head()


,customer_id,alquileres_05_2005
0,1,2
1,2,1
2,3,2
3,5,3
4,6,3


In [234]:
df_alquileres_2_2006 = rental_counth_month(df_rentals_2_2006,2,2006) 
df_alquileres_2_2006.head()


,customer_id,alquileres_02_2006
0,5,1
1,9,1
2,11,1
3,14,1
4,15,2


Crea una función en Python compare_rentals que reciba como entrada dos DataFrames que contengan el número de alquileres realizados por cada cliente en diferentes meses y años.   
La función debe devolver un DataFrame combinado con una nueva columna llamada 'diferencia', que representa la diferencia entre el número de alquileres en los dos meses.

In [235]:
def compare_rentals(df1, df2):
    df_merged = df1.merge(df2, on="customer_id", how="outer").fillna(0).astype(int)# por que piden data frame combinado y aprovecho y quito int
  
    col1 = df1.columns[1]
    col2 = df2.columns[1]

    df_merged["diferencia"] = (df_merged[col1] - df_merged[col2]).astype(int)

    return df_merged



In [236]:
compare_rentals(df_alquileres_5_2005,df_alquileres_2_2006)

,customer_id,alquileres_05_2005,alquileres_02_2006,diferencia
0,1,2,0,2
1,2,1,0,1
2,3,2,0,2
3,5,3,1,2
4,6,3,0,3
...,...,...,...,...
534,594,4,0,4
535,595,1,0,1
536,596,6,1,5
537,597,2,1,1
